# Project 21: Flight Delay Prediction and Rescheduling

## Task 3 — Leakage Audit and Feature Engineering

### Objective

The objective of this stage is to identify and prevent data leakage and create meaningful features that can improve flight delay prediction. The project aims to predict whether a flight will arrive more than 15 minutes late using information available before departure.

This notebook builds upon the cleaned dataset and exploratory data analysis performed in Task 2. It focuses on removing leakage-prone information, extracting useful time and route features, and calculating historical delay statistics using only past observations.

The engineered dataset will subsequently be used for chronological splitting, preprocessing, and model development.

### 1. Loading the Cleaned Dataset

We begin by importing the required Python libraries and loading the cleaned flight dataset generated during the previous stage. Pandas is used for data manipulation, while NumPy supports numerical operations and mathematical feature transformations.

We also convert the flight date column into a proper datetime format to enable chronological operations and time-based feature engineering. Finally, we inspect the dataset dimensions, available columns, date range, and target distribution to verify that the data has been loaded correctly.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("/content/flights_cleaned_FINAL.csv")

df = pd.read_csv(DATA_PATH)

print("Reading:", DATA_PATH)
print("Rows:", len(df))
print("File size:", DATA_PATH.stat().st_size)
print("Months:", sorted(df["month"].unique()))

Reading: /content/flights_cleaned_FINAL.csv
Rows: 327346
File size: 26576099
Months: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


### 2. Leakage Audit and Feature Availability

Data leakage occurs when a predictive model uses information that would not be available at the time a prediction is made. In this project, the prediction is assumed to occur before the flight departs.

Therefore, actual departure time, departure delay, arrival time, arrival delay, and actual flight duration must not be used as model predictors. The arrival delay is retained only as the basis for the target variable, `delayed`.

We also distinguish between useful pre-departure information, identifiers, and potentially unreliable features. This audit documents the reasoning behind each decision and establishes a consistent set of rules for the subsequent modelling stages.

In [ ]:
leakage_audit = pd.DataFrame([
    ("dep_time", "Actual departure time", "DROP"),
    ("dep_delay", "Actual departure delay", "DROP"),
    ("arr_time", "Actual arrival time", "DROP"),
    ("arr_delay", "Arrival delay used to define target", "DROP"),
    ("air_time", "Actual flight duration", "DROP"),
    ("year", "Constant year in this dataset", "DROP"),
    ("id", "Row identifier", "EXCLUDE FROM FEATURES"),
    ("name", "Duplicate carrier name", "EXCLUDE FROM FEATURES"),
    ("tailnum", "Aircraft assignment may change", "EXCLUDE INITIALLY"),
    ("flight", "Flight number; high cardinality", "OPTIONAL"),
    ("month", "Scheduled calendar month", "KEEP"),
    ("weekday", "Day of week", "KEEP"),
    ("hour", "Scheduled departure hour", "KEEP"),
    ("minute", "Scheduled departure minute", "KEEP"),
    ("sched_dep_time", "Scheduled departure time", "KEEP"),
    ("sched_arr_time", "Scheduled arrival time", "KEEP"),
    ("carrier", "Airline identity", "KEEP"),
    ("origin", "Origin airport", "KEEP"),
    ("dest", "Destination airport", "KEEP"),
    ("distance", "Route distance", "KEEP"),
], columns=["column", "meaning", "decision"])

display(leakage_audit)

leakage_audit.to_csv("03_leakage_audit.csv", index=False)

,column,meaning,decision
0,dep_time,Actual departure time,DROP
1,dep_delay,Actual departure delay,DROP
2,arr_time,Actual arrival time,DROP
3,arr_delay,Arrival delay used to define target,DROP
4,air_time,Actual flight duration,DROP
5,year,Constant year in this dataset,DROP
6,id,Row identifier,EXCLUDE FROM FEATURES
7,name,Duplicate carrier name,EXCLUDE FROM FEATURES
8,tailnum,Aircraft assignment may change,EXCLUDE INITIALLY
9,flight,Flight number; high cardinality,OPTIONAL


### 3. Verification of Data Quality and Leakage-Sensitive Columns

After documenting the leakage risks, we verify whether any excluded columns remain in the cleaned dataset. We also check missing values, duplicate rows, target values, and the validity of the date column.

These checks help identify potential data-quality issues before feature engineering begins. The target is expected to be binary, where 1 represents a flight arriving more than 15 minutes late and 0 represents all other flights classified under the project's target definition.

In [ ]:
forbidden_features = [
    "dep_time",
    "dep_delay",
    "arr_time",
    "arr_delay",
    "air_time",
    "year",
    "name",
    "status",
    "cancelled",
]

present = [col for col in forbidden_features if col in df.columns]

print("Leakage-sensitive or excluded columns still present:")
print(present)

print("\nMissing values:")
print(df.isna().sum()[lambda counts: counts > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget values:")
print(sorted(df["delayed"].unique()))

assert set(df["delayed"].unique()).issubset({0, 1})
assert df["date"].notna().all()

Leakage-sensitive or excluded columns still present:
[]

Missing values:
origin_rate_prev7d    1759
dtype: int64

Duplicate rows: 0

Target values:
[np.float64(0.0), np.float64(1.0)]


### 4. Temporal Feature Engineering Using Cyclic Encoding

Flight delays can vary with departure time, day of the week, month, and scheduled arrival time. However, these variables contain cyclic relationships that are not always represented well by ordinary numerical values.

For example, 23:00 and 00:00 are adjacent times, even though their numerical hour values differ considerably. To represent such periodic relationships, we transform scheduled departure time, weekday, month, and scheduled arrival time into sine and cosine features.

These transformations preserve the cyclical nature of time and provide useful numerical inputs for machine learning models.

In [ ]:
# Scheduled departure time in minutes after midnight
df["dep_minute_of_day"] = (
    pd.to_numeric(df["hour"], errors="coerce") * 60
    + pd.to_numeric(df["minute"], errors="coerce")
)

# Cyclic departure time
df["dep_time_sin"] = np.sin(
    2 * np.pi * df["dep_minute_of_day"] / 1440
)
df["dep_time_cos"] = np.cos(
    2 * np.pi * df["dep_minute_of_day"] / 1440
)

# Cyclic weekday: Monday = 0, Sunday = 6
df["weekday_sin"] = np.sin(
    2 * np.pi * df["weekday"] / 7
)
df["weekday_cos"] = np.cos(
    2 * np.pi * df["weekday"] / 7
)

# Cyclic month: January = 1, December = 12
df["month_sin"] = np.sin(
    2 * np.pi * (df["month"] - 1) / 12
)
df["month_cos"] = np.cos(
    2 * np.pi * (df["month"] - 1) / 12
)

# Validate scheduled arrival HHMM values
arr_raw = pd.to_numeric(
    df["sched_arr_time"], errors="coerce"
)

valid_arr = (
    arr_raw.notna()
    & arr_raw.ge(0)
    & arr_raw.le(2400)
    & (arr_raw % 100 <= 59)
    & ((arr_raw // 100 <= 23) | (arr_raw == 2400))
)

# Convert HHMM to minutes after midnight.
# 2400 represents midnight (00:00).
arr_hour = arr_raw // 100
arr_minute = arr_raw % 100

df["arr_minute_of_day"] = (
    arr_hour * 60 + arr_minute
).where(valid_arr)

df.loc[arr_raw == 2400, "arr_minute_of_day"] = 0

df["arr_time_sin"] = np.sin(
    2 * np.pi * df["arr_minute_of_day"] / 1440
)
df["arr_time_cos"] = np.cos(
    2 * np.pi * df["arr_minute_of_day"] / 1440
)

print("Time features created.")
print("Invalid scheduled arrival times:", (~valid_arr).sum())

Time features created.
Invalid scheduled arrival times: 0


### 5. Route and Operational Feature Engineering

Flight delay behaviour can vary across airlines, departure airports, destination airports, and individual routes. Therefore, we retain carrier, origin, destination, and distance as potential predictors.

We also create a combined route identifier using the origin and destination airports. This allows the model to distinguish between different origin-destination combinations rather than treating each airport independently.

These features provide a foundation for learning route-specific and airline-specific delay patterns. Categorical encoding and any additional preprocessing will be handled in the subsequent preprocessing stage.

In [ ]:
# Combined route identifier
df["route"] = (
    df["origin"].astype(str)
    + "_"
    + df["dest"].astype(str)
)

# Route direction is preserved:
# JFK_LAX and LAX_JFK are different routes.

print("Unique carriers:", df["carrier"].nunique())
print("Unique origins:", df["origin"].nunique())
print("Unique destinations:", df["dest"].nunique())
print("Unique routes:", df["route"].nunique())

display(
    df[[
        "carrier",
        "origin",
        "dest",
        "route",
        "distance",
        "hour",
        "sched_arr_time"
    ]].head()
)

Unique carriers: 16
Unique origins: 3
Unique destinations: 104
Unique routes: 223


,carrier,origin,dest,route,distance,hour,sched_arr_time
0,UA,EWR,IAH,EWR_IAH,1400,5,819
1,UA,LGA,IAH,LGA_IAH,1416,5,830
2,AA,JFK,MIA,JFK_MIA,1089,5,850
3,B6,JFK,BQN,JFK_BQN,1576,5,1022
4,DL,LGA,ATL,LGA_ATL,762,6,837


### 6. Historical Delay Feature Engineering

Historical flight performance can provide useful information about recurring delay patterns at airports, across airlines, and on particular routes. To capture these patterns, we calculate historical delay rates and flight counts for each category.

A critical requirement is that historical features must use only information available before the flight's prediction time. We therefore aggregate observed outcomes by category and date and apply a rolling window that excludes the current date.

The selected window covers the seven preceding calendar dates. This approach prevents the historical statistics for a flight from incorporating delay outcomes recorded on the same date or on future dates.

**Operational cutoff:** For conservative pre-departure forecasting, historical outcomes from the prediction day and immediately preceding calendar day are excluded. For date D, the rate uses days D−8 through D−2. Historical rates remain missing when the historical flight count is zero; training-only imputation is required later. This lag is an assumption, not proof that every observation meets an exact 24-hour cutoff.

In [ ]:

def add_past_7d_features(data, group_col):
    """
    Historical aggregates for calendar days D-8 through D-2.
    Excludes current day and immediately previous day
    to reduce leakage in pre-departure predictions.
    """

    daily = data.groupby(
        [group_col, 'date'], as_index=False
    ).agg(
        delay_sum=('delayed', 'sum'),
        flight_count=('delayed', 'size')
    )

    parts = []

    for group_value, g in daily.groupby(group_col, sort=False):

        g = g.set_index('date').sort_index()

        # Create continuous daily calendar
        calendar = pd.date_range(
            g.index.min(),
            g.index.max(),
            freq="D"
        )

        # Fill dates with no flights using zero counts
        full = g[
            ["delay_sum", "flight_count"]
        ].reindex(
            calendar,
            fill_value=0
        )

        # Previous seven calendar days, with 2-day lag
        past = (
            full.shift(2, fill_value=0)
                .rolling(window=7, min_periods=1)
                .sum()
        )

        # Match historical values to flight dates
        observed = past.reindex(g.index)

        part = pd.DataFrame({
            group_col: group_value,
            'date': g.index,

            f'{group_col}_flights_prev7d':
                observed['flight_count'].to_numpy(),

            f'{group_col}_delay_rate_prev7d': (
                observed['delay_sum']
                / observed['flight_count'].replace(0, np.nan)
            ).to_numpy()
        })

        parts.append(part)

    history = pd.concat(parts, ignore_index=True)

    return data.merge(
        history,
        on=[group_col, 'date'],
        how='left',
        validate='many_to_one'
    )


### 7. Generating Historical Features for Airports, Carriers, and Routes

Using the historical feature function defined above, we generate past-seven-day delay rates and flight counts for four categories: origin airport, airline carrier, destination airport, and origin-destination route.

The delay rate represents the proportion of previously observed flights in the relevant category that were delayed. The corresponding flight count indicates how many historical observations contributed to that statistic.

When no prior observations are available, the historical rate is filled with zero and the historical count remains zero. This provides a consistent numerical representation while allowing the model to distinguish between a zero rate based on available history and the absence of historical observations.

In [ ]:
for col in ["origin", "carrier", "dest", "route"]:
    df = add_past_7d_features(df, col)

historical_cols = [
    "origin_delay_rate_prev7d",
    "origin_flights_prev7d",
    "carrier_delay_rate_prev7d",
    "carrier_flights_prev7d",
    "dest_delay_rate_prev7d",
    "dest_flights_prev7d",
    "route_delay_rate_prev7d",
    "route_flights_prev7d",
]

display(df[["date", "carrier", "origin", "dest", "route",
            "delayed"] + historical_cols].head(15))

,date,carrier,origin,dest,route,delayed,origin_delay_rate_prev7d,origin_flights_prev7d,carrier_delay_rate_prev7d,carrier_flights_prev7d,dest_delay_rate_prev7d,dest_flights_prev7d,route_delay_rate_prev7d,route_flights_prev7d
0,2013-01-01,UA,EWR,IAH,EWR_IAH,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
1,2013-01-01,UA,LGA,IAH,LGA_IAH,1.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
2,2013-01-01,AA,JFK,MIA,JFK_MIA,1.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
3,2013-01-01,B6,JFK,BQN,JFK_BQN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
4,2013-01-01,DL,LGA,ATL,LGA_ATL,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
5,2013-01-01,UA,EWR,ORD,EWR_ORD,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
6,2013-01-01,B6,EWR,FLL,EWR_FLL,1.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
7,2013-01-01,EV,LGA,IAD,LGA_IAD,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
8,2013-01-01,B6,JFK,MCO,JFK_MCO,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
9,2013-01-01,AA,LGA,ORD,LGA_ORD,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0


### 8. Validation of Historical Features

After generating the historical statistics, we validate that all delay rates fall between 0 and 1 and that the historical flight counts are non-negative.

These checks help confirm that the engineered values are mathematically valid. We also inspect descriptive statistics to understand the distribution of the historical features and identify any unexpected values before moving to the final feature selection stage.

In [ ]:
# All historical rates must be between 0 and 1.
for col in [
    "origin_delay_rate_prev7d",
    "carrier_delay_rate_prev7d",
    "dest_delay_rate_prev7d",
    "route_delay_rate_prev7d",
]:
    assert df[col].dropna().between(0, 1).all()

# Historical flight counts must be non-negative.
for col in [
    "origin_flights_prev7d",
    "carrier_flights_prev7d",
    "dest_flights_prev7d",
    "route_flights_prev7d",
]:
    assert (df[col] >= 0).all()

print("Historical feature checks passed.")

display(
    df[historical_cols].describe().round(3)
)

Historical feature checks passed.


,origin_delay_rate_prev7d,origin_flights_prev7d,carrier_delay_rate_prev7d,carrier_flights_prev7d,dest_delay_rate_prev7d,dest_flights_prev7d,route_delay_rate_prev7d,route_flights_prev7d
count,0.0,327346.0,0.0,327346.0,0.0,327346.0,0.0,327346.0
mean,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
std,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
min,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
25%,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
50%,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
75%,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0
max,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0


In [ ]:
print("Unique dates:", df["date"].nunique())
print("First date:", df["date"].min())
print("Last date:", df["date"].max())
print("Missing dates:", df["date"].isna().sum())

Unique dates: 365
First date: 2013-01-01
Last date: 2013-12-31
Missing dates: 0


### 9. Defining the Recommended Model Feature Set

We now define the recommended predictor variables for the subsequent modelling stages. The selected features include cyclic temporal variables, route information, distance, and historical delay statistics.

Categorical features are maintained separately from numerical features because different models require different preprocessing strategies. For example, Logistic Regression may use one-hot encoding, while a deep learning model can use learned embeddings for categorical variables.

The target variable, flight date, identifiers, and split labels are excluded from the predictor list. They may still be retained in the dataset for evaluation, record identification, or chronological splitting, but must not be passed to the models as ordinary input features.

In [ ]:
categorical_features = [
    "carrier",
    "origin",
    "dest",
    "route"
]

numeric_features = [
    "distance",

    "dep_time_sin",
    "dep_time_cos",

    "weekday_sin",
    "weekday_cos",

    "month_sin",
    "month_cos",

    "arr_time_sin",
    "arr_time_cos",

    "origin_delay_rate_prev7d",
    "origin_flights_prev7d",

    "carrier_delay_rate_prev7d",
    "carrier_flights_prev7d",

    "dest_delay_rate_prev7d",
    "dest_flights_prev7d",

    "route_delay_rate_prev7d",
    "route_flights_prev7d",
]

model_features = categorical_features + numeric_features

target = "delayed"

# These columns are retained for identification or splitting,
# but are never passed as model predictors.
metadata_columns = [
    "id",
    "date",
    "split",
    "delayed",
    "hour",
    "minute",
    "month",
    "day",
    "weekday",
    "sched_dep_time",
    "sched_arr_time",
    "dep_minute_of_day",
    "sched_arr_hour",
    "sched_arr_minute",
    "arr_minute_of_day",
    "flight",
    "tailnum",
]

assert len(model_features) == len(set(model_features))
assert target not in model_features
assert "id" not in model_features
assert "date" not in model_features
assert "split" not in model_features

print("Number of categorical features:", len(categorical_features))
print("Number of numeric features:", len(numeric_features))
print("Total recommended model features:", len(model_features))

print("\nCategorical features:", categorical_features)
print("\nNumeric features:", numeric_features)

Number of categorical features: 4
Number of numeric features: 17
Total recommended model features: 21

Categorical features: ['carrier', 'origin', 'dest', 'route']

Numeric features: ['distance', 'dep_time_sin', 'dep_time_cos', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'arr_time_sin', 'arr_time_cos', 'origin_delay_rate_prev7d', 'origin_flights_prev7d', 'carrier_delay_rate_prev7d', 'carrier_flights_prev7d', 'dest_delay_rate_prev7d', 'dest_flights_prev7d', 'route_delay_rate_prev7d', 'route_flights_prev7d']


### 10. Feature Dictionary and Documentation

A feature dictionary provides a structured description of the engineered variables, their types, and their intended meanings. This improves the interpretability and reproducibility of the project.

The dictionary also records whether each variable is included in the recommended model feature set. It serves as a reference for the team during preprocessing, model development, debugging, and final report preparation.

In [ ]:
feature_dictionary = pd.DataFrame([
    ("dep_time_sin", "Numeric", "Cyclic scheduled departure time"),
    ("dep_time_cos", "Numeric", "Cyclic scheduled departure time"),
    ("weekday_sin", "Numeric", "Cyclic day of week"),
    ("weekday_cos", "Numeric", "Cyclic day of week"),
    ("month_sin", "Numeric", "Cyclic calendar month"),
    ("month_cos", "Numeric", "Cyclic calendar month"),
    ("arr_time_sin", "Numeric", "Cyclic scheduled arrival time"),
    ("arr_time_cos", "Numeric", "Cyclic scheduled arrival time"),
    ("distance", "Numeric", "Scheduled route distance"),
    ("carrier", "Categorical", "Airline identifier"),
    ("origin", "Categorical", "Departure airport"),
    ("dest", "Categorical", "Destination airport"),
    ("route", "Categorical", "Origin-destination combination"),
    ("origin_delay_rate_prev7d", "Historical", "Prior seven calendar days' origin delay rate"),
    ("origin_flights_prev7d", "Historical", "Prior seven calendar days' origin flight count"),
    ("carrier_delay_rate_prev7d", "Historical", "Prior seven calendar days' carrier delay rate"),
    ("carrier_flights_prev7d", "Historical", "Prior seven calendar days' carrier flight count"),
    ("dest_delay_rate_prev7d", "Historical", "Prior seven calendar days' destination delay rate"),
    ("dest_flights_prev7d", "Historical", "Prior seven calendar days' destination flight count"),
    ("route_delay_rate_prev7d", "Historical", "Prior seven calendar days' route delay rate"),
    ("route_flights_prev7d", "Historical", "Prior seven calendar days' route flight count"),
], columns=["feature", "type", "description"])

feature_dictionary["included_in_recommended_model"] = (
    feature_dictionary["feature"].isin(model_features)
)

display(feature_dictionary)

feature_dictionary.to_csv(
    "03_feature_dictionary.csv",
    index=False
)

,feature,type,description,included_in_recommended_model
0,dep_time_sin,Numeric,Cyclic scheduled departure time,True
1,dep_time_cos,Numeric,Cyclic scheduled departure time,True
2,weekday_sin,Numeric,Cyclic day of week,True
3,weekday_cos,Numeric,Cyclic day of week,True
4,month_sin,Numeric,Cyclic calendar month,True
5,month_cos,Numeric,Cyclic calendar month,True
6,arr_time_sin,Numeric,Cyclic scheduled arrival time,True
7,arr_time_cos,Numeric,Cyclic scheduled arrival time,True
8,distance,Numeric,Scheduled route distance,True
9,carrier,Categorical,Airline identifier,True


### 11. Final Validation and Export

In the final stage, we perform consistency checks on the engineered dataset. We verify that the target and existing chronological split labels are valid, that the recommended model features are present, and that leakage-sensitive columns have not been retained as predictors.

The dataset is then sorted chronologically and exported alongside the feature dictionary, leakage audit, and recommended feature list. These outputs provide a reproducible handoff to the next stage, which will perform chronological validation and preprocessing before model training.

The resulting feature set is intended to support fair model comparison using only information available at the selected pre-departure prediction time.

In [ ]:
# Validate the complete source dataset is intact.
assert len(df) == 327346, f'Expected 327346 rows; got {len(df)}'
assert df['date'].nunique() == 365
assert not df.duplicated().any(), 'Unexpected duplicate rows'
# Sort chronologically for reproducibility.
sort_cols = ["date"]

if "id" in df.columns:
    sort_cols.append("id")

df = df.sort_values(sort_cols).reset_index(drop=True)

# Validate target and split labels.
assert df["delayed"].notna().all()
assert set(df["delayed"].unique()).issubset({0, 1})
assert df["date"].notna().all()

assert df["split"].isin(
    ["train", "val", "test"]
).all()

# Confirm no post-departure leakage columns remain.

# Confirm no post-departure leakage columns are model inputs.
forbidden = [
    "dep_time",
    "dep_delay",
    "arr_time",
    "arr_delay",
    "air_time",
    "year",
    "name",
    "status",
    "cancelled",
]

present_forbidden = [
    col for col in forbidden if col in model_features
]

assert not present_forbidden, (
    f"Leakage-sensitive model inputs found: {present_forbidden}"
)

print("Model feature leakage check passed.")


# Confirm all recommended model inputs exist.
missing_features = [
    col for col in model_features if col not in df.columns
]

assert not missing_features, (
    f"Missing model features: {missing_features}"
)
expected_splits = {"train", "val", "test"}
actual_splits = set(df["split"].dropna().unique())

assert actual_splits == expected_splits, (
    f"Expected {expected_splits}, found {actual_splits}"
)
# Check that the date ranges do not overlap between splits.
split_dates = (
    df.groupby("split")["date"]
    .agg(["min", "max"])
)

display(split_dates)

train_max = split_dates.loc["train", "max"]
val_min = split_dates.loc["val", "min"]
val_max = split_dates.loc["val", "max"]
test_min = split_dates.loc["test", "min"]

assert train_max < val_min, (
    "Training and validation dates overlap or are out of order."
)

assert val_max < test_min, (
    "Validation and test dates overlap or are out of order."
)

print("Chronological split checks passed.")
assert len(model_features) == len(set(model_features))
assert target not in model_features
assert "id" not in model_features
assert "date" not in model_features
assert "split" not in model_features

# Export the engineered dataset.
# Export only metadata needed for evaluation and explicitly allowed predictors.
export_cols = ['date', 'split', 'delayed'] + model_features
assert len(export_cols) == len(set(export_cols))
assert not df[model_features].columns.isin(forbidden).any()
export_df = df[export_cols].copy()
export_df.to_csv('flights_feature_engineered.csv', index=False)

pd.Series(model_features, name="feature").to_csv(
    "03_model_features.csv",
    index=False
)

print("Feature engineering complete!")
print("Output shape:", df.shape)

print("\nSplit distribution:")
print(df["split"].value_counts())

print("\nSaved:")
print("1. flights_feature_engineered.csv")
print("2. 03_leakage_audit.csv")
print("3. 03_feature_dictionary.csv")
print("4. 03_model_features.csv")
print('Export shape:', export_df.shape)
print('Missing feature values (to impute using train only):')
print(export_df[model_features].isna().sum().loc[lambda s: s > 0])

Model feature leakage check passed.


,min,max
split,,
test,2013-11-01,2013-12-31
train,2013-01-01,2013-08-31
val,2013-09-01,2013-10-31


Chronological split checks passed.
Feature engineering complete!
Output shape: (327346, 35)

Split distribution:
split
train    217727
val       55628
test      53991
Name: count, dtype: int64

Saved:
1. flights_feature_engineered.csv
2. 03_leakage_audit.csv
3. 03_feature_dictionary.csv
4. 03_model_features.csv
Export shape: (327346, 24)
Missing feature values (to impute using train only):
origin_delay_rate_prev7d     327346
carrier_delay_rate_prev7d    327346
dest_delay_rate_prev7d       327346
route_delay_rate_prev7d      327346
dtype: int64
